In [2]:
from get_b3_data import B3PublicData
import pandas as pd
import numpy as np
import glob
import os
pd.set_option('display.max_columns', None)

In [3]:
# COLS = ['DATA',	'CODBDI', 'TICKER', 'TPMERC', 'NOMRES', 'ABERTURA', 'MAXIMO', 'MINIMO', 'FECHAMENTO', 'VOL_TOTAL']
# anos = range(2015, 2026)
# b3 = B3PublicData()

# for ano in anos:
#     df = b3.read_df_year(ano)
#     os.makedirs(f"raw/data/{ano}", exist_ok=True)
#     df[COLS].to_parquet(f"data/raw/{ano}/cotahist_{ano}.parquet")

In [4]:
# pastas = os.listdir("data/raw/")
# dfs = []
# for pasta in pastas:
#     file = f'data/raw/{pasta}/cotahist_{pasta}.parquet'
#     dfs.append(pd.read_parquet(file))

In [5]:
# df_base = pd.concat(dfs)a
df_base = pd.read_parquet('data/processed/df_base.parquet')
df_base.loc[df_base['TICKER'] == 'CCRO3', 'TICKER'] = 'MOTV3'
df_base.loc[df_base['TICKER'] == 'CVBI11', 'TICKER'] = 'PCIP11'
df_base.loc[df_base['TICKER'] == 'ELET3', 'TICKER'] = 'AXIA3'
df_base.loc[df_base['TICKER'] == 'ELET6', 'TICKER'] = 'AXIA6'
df_base.loc[df_base['TICKER'] == 'EMBR3', 'TICKER'] = 'EMBJ3'
df_base.loc[df_base['TICKER'] == 'JBSS3', 'TICKER'] = 'JBSS32'
df_base.loc[df_base['TICKER'] == 'LVTC3', 'TICKER'] = 'WDCN3'
df_base.loc[df_base['TICKER'] == 'MALL11', 'TICKER'] = 'PMLL11'
df_base.loc[df_base['TICKER'] == 'MBLY3', 'TICKER'] = 'TOKY3'
df_base.loc[df_base['TICKER'] == 'NCHB11', 'TICKER'] = 'FYTO11'
df_base.loc[df_base['TICKER'] == 'NINJ3', 'TICKER'] = 'REAG3'
df_base.loc[df_base['TICKER'] == 'NTCO3', 'TICKER'] = 'NATU3'
df_base.loc[df_base['TICKER'] == 'OULG11', 'TICKER'] = 'TRUE11'
df_base.loc[df_base['TICKER'] == 'RBFF11', 'TICKER'] = 'RBFM11'
df_base.loc[df_base['TICKER'] == 'QAGR11', 'TICKER'] = 'PLAG11'
df_base.loc[df_base['TICKER'] == 'RMAI11', 'TICKER'] = 'LMAI11'
df_base.loc[df_base['TICKER'] == 'RRCI11', 'TICKER'] = 'XLPR11'
df_base.loc[df_base['TICKER'] == 'RVBI11', 'TICKER'] = 'PSEC11'
df_base.loc[df_base['TICKER'] == 'XPPR11', 'TICKER'] = 'VPPR11'
df_base.loc[df_base['TICKER'] == 'REAG3', 'TICKER'] = 'ARND3'

In [6]:
#df_base[df_base['TICKER'] == 'ARND3']

In [7]:
df_base.to_parquet('data/processed/df_base.parquet')

In [8]:
# lendo dados financeiros de 2025 para pegar os tickers que existem
# e depois filtrar para manter apenas os que existem
tickers_2025 = pd.read_parquet('data/raw/2025/cotahist_2025.parquet')['TICKER'].unique()

In [9]:
# pegando empresas com menos de 252 [1 ano de negociacao] linhas de negociação para remover
di = df_base.groupby('TICKER').size()
di = di[di <= 252].index.tolist()
# pegando tickers de empresas que entraram em recuperação judicial para remover
ticker_rec_judicial = df_base[df_base['CODBDI'] == 8]['TICKER'].unique()
tickers_dados_insuf = [
    'ADMF3', 'AETH39', 'ANGV39', 'APXR11', 'APXU11', 'ASMT11', 'ASRF11', 'AXIA5', 'AXRP39', 'BAOM39',
    'BARY39', 'BASK39', 'BBAJ39', 'BBCN39', 'BBER39', 'BBGR39', 'BBIL39', 'BBJP39', 'BBMR39'
] + di
tickers_saiu_bolsa = [
    'AZUL4', 'BPAN4', 'BPFF11', 'BRFS3', 'BTEK11', 'CCRO3', 'CLSA3', 'CPLE5', 'CPLE6', 
    'CRFB3', 'ELMD3', 'EXCO32', 'GOLL4', 'GPIV33', 'KRSA3', 'LIPR3', 'MOAR3', 'NFTS11', 
    'PORT3', 'PULV11', 'RBED11', 'RDNI3', 'SRNA3', 'STBP3', 'STOC31', 'ZAMP3',
]
df_database = df_base[
    ~df_base['TICKER'].str.endswith('34') &
    (df_base['CODBDI'].isin([2, 12, 14, 34, 35, 36])) &
    (df_base['TICKER'].isin(tickers_2025)) &
    ~(df_base['TICKER'].isin(tickers_dados_insuf)) &
    ~(df_base['TICKER'].isin(ticker_rec_judicial)) &
    ~(df_base['TICKER'].isin(tickers_saiu_bolsa))
]

In [10]:
def to_yfinance_style(
    df: pd.DataFrame,
    ticker: str | None = None,
    adjust_prices: bool = True,
) -> pd.DataFrame:
    RENAME = {
        "ABERTURA": "Open",
        "MAXIMO": "High",
        "MINIMO": "Low",
        "FECHAMENTO": "Close",
        "VOL_TOTAL": "Volume",
    }
    METRICS_ORDER = ["Close", "High", "Low", "Open", "Volume"]
    PRICE_COLS = ["Open", "High", "Low", "Close"]

    cols_needed = ["DATA", "TICKER"] + list(RENAME.keys())
    out = df[cols_needed].copy()
    out["TICKER"] = out["TICKER"].astype(str).str.upper()

    out["Date"] = pd.to_datetime(out["DATA"], format="%Y%m%d")
    out = out.drop(columns=["DATA"]).rename(columns=RENAME)

    if adjust_prices:
        out[PRICE_COLS] = out[PRICE_COLS] / 100

    out = out.set_index("Date").sort_index()
    out = out.pivot_table(
        index="Date",
        columns="TICKER",
        values=METRICS_ORDER,
        aggfunc="first",  # ou "mean" se preferir agregar duplicatas
    )
    out = out.sort_index(axis=1, level=1)
    out.columns.names = ["Price", "Ticker"]
    out.index.name = "Date"
    return out

df_final = to_yfinance_style(df_database)

In [11]:
df_final.to_parquet('data/processed/data_assets.parquet')

In [12]:
k = pd.read_parquet('data/processed/data_assets.parquet')
# Calcular retornos log (mais estável que preço bruto)
returns = np.log(k['Close'] / k['Close'].shift(1))
# Correlação — pandas já ignora NaN pairwise
corr_matrix = returns.corr(method='pearson')

c:\Users\kimbo\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


In [13]:
corr_matrix.to_parquet('data/processed/corr_matrix.parquet')

In [14]:
import yfinance as yf
import time
from requests.exceptions import HTTPError

cm = pd.read_parquet('data/processed/corr_matrix.parquet')
tickers = cm.columns.tolist()
setores = []


def to_yahoo_symbol(ticker: str) -> str:
    t = str(ticker).strip().upper()
    if t.endswith(".SA"):
        return t
    return f"{t}.SA"


for ticker in tickers:
    yahoo_sym = to_yahoo_symbol(ticker)
    try:
        empresa = yf.Ticker(yahoo_sym)
        setor = empresa.info.get("sector")
    except HTTPError as e:
        print(f"Erro ao obter o setor para {ticker} ({yahoo_sym}): {e}")
        setor = None
    except Exception as e:
        print(f"Erro ao obter o setor para {ticker} ({yahoo_sym}): {e}")
        setor = None
    setores.append({"ticker": ticker, "setor": setor, "yahoo_symbol": yahoo_sym}) 
    time.sleep(0.5)

HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 
HTTP Error 404: 


In [15]:
# removidos da bolsa
# AZUL4	
# BPAN4	
# BPFF11
# BRFS3	
# BTEK11
# CCRO3	
# CLSA3
# CPLE5
# CPLE6
# CRFB3
# ELMD3
# EXCO32
# GOLL4
# GPIV33
# KRSA3
# LIPR3
# MOAR3
# NFTS11
# PORT3
# PULV11
# RBED11
# RDNI3
# SRNA3
# STBP3
# STOC31
# ZAMP3

In [16]:
df_setores = pd.DataFrame(setores)
mask_etf = df_setores['ticker'].str.endswith('39')
df_setores.loc[mask_etf, 'setor'] = 'ETF'
mask_fii = df_setores['ticker'].str.endswith('11')
df_setores.loc[mask_fii, 'setor'] = 'Real Estate'
df_setores.loc[df_setores['ticker'] == 'FOOD11', 'setor'] = 'ETF'
df_setores.loc[df_setores['ticker'] == 'GUAR3', 'setor'] = 'Apparel Manufacturing'
df_setores.loc[df_setores['ticker'] == 'PETZ3', 'setor'] = 'Personal Services'
df_setores.loc[df_setores['ticker'] == 'PSVM11', 'setor'] = 'Marine Shipping'
df_setores.loc[df_setores['ticker'] == 'VSTE3', 'setor'] = 'Apparel Retail'
df_setores.loc[df_setores['ticker'] == 'MRFG3', 'setor'] = 'Packaged Foods'
df_setores.loc[df_setores['ticker'] == 'EVTC31', 'setor'] = 'Technology'

df_setores[df_setores['setor'].isna()]

,ticker,setor,yahoo_symbol


In [17]:
df_setores.to_parquet('data/processed/df_setores.parquet')